# Build your hierarchical agent

In this tutorial we'll build our first 2-level hierarchical agent with TAG.

We'll see how to configure agents, how to configure levels and how to stack them. Finally we'll also train the resulting hierarchy and test it.

## Environment

We'll use the [MPE Simple Spread environment](https://pettingzoo.farama.org/environments/mpe/simple_spread/) from Pettingzoo with 2 agents.

Note that we also have our own wrapper for the environment, that colors the agents to help with visualization: `src/tame/envs/mpe_simple_spread/env.py`. But in this example we'll use the standard one.

In [1]:
from pettingzoo.mpe import simple_spread_v3

TOTAL_AGENTS = 2
# Note that we use the ParallelEnv here, as the library only supports that one for now.
env = simple_spread_v3.parallel_env(N=TOTAL_AGENTS,max_cycles=100,continuous_actions=False, render_mode='rgb_array')

In [2]:
env_obs_sizes = []
for agent in env.observation_spaces:
    obs_space = env.observation_spaces[agent]
    env_obs_sizes.append(obs_space.shape[0])

/mnt/data/gpaolo/miniforge3/envs/tame/lib/python3.10/site-packages/pettingzoo/utils/conversions.py:144: UserWarning: The `observation_spaces` dictionary is deprecated. Use the `observation_space` function instead.
  warnings.warn(


# Hierarchy structure

We'll define our 2 level hierarchy with PPO-based subagents.

We start by defining the agent names and how they are connected:

In [3]:
# TODO rename agent to ppo...
bottom_agent_names = [f"agent_{i}" for i in range(len(env.possible_agents))]
top_agent_name = "top_ppo"

In [4]:
# Each bottom agent is connected to one agent in the environment
bottom_env_links = {agent_name: [env_agent] for agent_name, env_agent in zip(bottom_agent_names, env.possible_agents)}
print(f"Bottom to env connections: {bottom_env_links}")

# The top agent is connected to the two bottome agents
top_bottom_links = {top_agent_name: bottom_agent_names}
print(f"Top to bottom connections: {top_bottom_links}")

Bottom to env connections: {'agent_0': ['agent_0'], 'agent_1': ['agent_1']}
Top to bottom connections: {'top_ppo': ['agent_0', 'agent_1']}


We now instantiate the hierarchy itself, that will help us organize the levels and the communications:

In [5]:
from tame.hierarchy import Hierarchy

hierarchy = Hierarchy()

## Bottom level
### Agents

Now we can parametrize the agents in the bottom level.

For this we use `AgentConfig`, which allows us to provide the configuration for each of the agents in the level.

In [6]:
import numpy as np
from pprint import pprint

from gymnasium.spaces import Dict as GymDict, Box, Discrete
from tame.hierarchy import AgentConfig
# We use our PPO implementation as agent in the level. Note that this is a LevelAgent.
from tame.agents.monolithic_ppo import Agent as PPO 
from tame.agents.monolithic_ppo import Args as PPOArgs 

An important aspect of the agents definition is the communication space.
There are multiple ways agents can communicate to the level above in TAG (no communication, just pass the observations from below, learn to communicate.)

In this simple agent, we just pass the observations from the level below.
This means that the communication space of each agent in the bottome level is equal to the observation space for each agent in the environment.

In [7]:

bottom_agents = []
for agent_name in bottom_agent_names:

    # We first define the communication space that our agent uses.
    # In this case we don't learn the communication space, but just have the agent concatenate the observations
    # it receives from the level below (that is the environment) from the agents it's connected with.
    # ---------------------------
    # Get list of agents from env connected to this agent
    env_agents = bottom_env_links[agent_name]
    print(f"Agent {agent_name} is connected to {env_agents}")

    # Get observation shapes of the connected agents
    env_observation_shapes = [env.observation_spaces[env_agent].shape for env_agent in env_agents]
    print(f"Observation shapes of agents connected to {agent_name}: {env_observation_shapes}")

    # This is the size of the message that the agent sends to the level above
    concatenated_obs_size = np.sum(env_observation_shapes)

    # Define agent communication space
    agent_comm_space = GymDict({agent_name: Box(np.inf, np.inf, shape=[concatenated_obs_size])})
    print(f"Agent {agent_name} communication space: {agent_comm_space}")
    # ---------------------------

    # Make agent config
    agent_config = AgentConfig(
        name=agent_name,
        agent_class=PPO,
        agent_kwargs={'args': PPOArgs()},
        communication_space=agent_comm_space
        )
    
    print(f"Agent {agent_name} config:")
    pprint(agent_config)
    print("===============================================")
    bottom_agents.append(agent_config)


Agent agent_0 is connected to ['agent_0']
Observation shapes of agents connected to agent_0: [(12,)]
Agent agent_0 communication space: Dict('agent_0': Box(inf, inf, (12,), float32))
Agent agent_0 config:
AgentConfig(name='agent_0',
            communication_space=Dict('agent_0': Box(inf, inf, (12,), float32)),
            agent_class=<class 'tame.agents.monolithic_ppo.Agent'>,
            agent_kwargs={'args': Args(subargs=None,
                                       seed=1,
                                       cuda=0,
                                       exp_name='monolithic_ppo',
                                       torch_deterministic=True,
                                       save_model=True,
                                       total_timesteps=500000,
                                       learning_rate=0.00025,
                                       gamma=0.99,
                                       anneal_lr=True,
                                       gae_lambda=0.95

/mnt/data/gpaolo/miniforge3/envs/tame/lib/python3.10/site-packages/pettingzoo/utils/conversions.py:144: UserWarning: The `observation_spaces` dictionary is deprecated. Use the `observation_space` function instead.
  warnings.warn(


### Level

Once we have the agents configurations, we can configure the level itself.

As for the agents it's important to define the communication space, for the level it's important to define the action space, that is which actions the level expects from the agents in the level above.

The action space is a nested Dict gym space the defines what the agents from top sends to which agent in the level.
It will have form:
{top_agent_name: {bottom_agent: agent_action_space}}

In this case, we'll have a discrete action space of size 5 for each connection.

In [8]:
bottom_lev_action_space = {}
for hl_agent, agent_names in top_bottom_links.items():
    agent_spaces = {agent_name: Discrete(5) for agent_name in agent_names}
    bottom_lev_action_space[hl_agent] = GymDict(agent_spaces)
bottom_lev_action_space = GymDict(bottom_lev_action_space)

print(bottom_lev_action_space)

Dict('top_ppo': Dict('agent_0': Discrete(5), 'agent_1': Discrete(5)))


This means that the `top_ppo` will send a separate `Discrete(5)` action to each of the agents in the `bottom` level.

Now, we have all the ingredients finally parametrize the whole level.
To do so we'll use the `LevelConfig` class to which we pass all the parameters.

In [9]:
from tame.hierarchy import LevelConfig

bottom_level_config = LevelConfig(
                name="bottom",
                agents=bottom_agents,
                uplinks=top_bottom_links, # How the agents in this level are connected to the level above
                downlinks=bottom_env_links, # How the agents in this level are connected to the level below
                action_frequency=1, # How often the agents in this level can act wrt to the level below
                trace_type="full", # 'full' or 'reward'. Full saves everything, reward saves only the rewards
                concat_obs=True, # If true, the actions from the level above are concatenated to the observations from the level below before being sent to the agents in this level
                action_space=bottom_lev_action_space, # type: ignore
                env=env, # The environment that the agents in this level interact with
            )

In [10]:
# Finally, we add the level to the hierarchy
hierarchy.add_level_config(bottom_level_config)

Created level bottom with agents: ['agent_0', 'agent_1']


#### Note
If you want to instantiate the level directly, you can use the `LevelEnv` from `src/tame/hierarchy/level_env.py` and then add it to the hierarchy through the `add_level()` function.


## Top level

Now we can define the top level of the hierarchy.

### Agent
As before, we define the top agent as PPO through the `AgentConfig`.
Note that here we don't have to make any communication space, as there is no level above!

In [11]:
top_agent_config = AgentConfig(
            name=top_agent_name,
            communication_space=None, # No communication space for the top agent
            agent_class=PPO,
            agent_kwargs={"args": PPOArgs()},
        )
print(f"Agent {top_agent_name} config:")
pprint(top_agent_config)

Agent top_ppo config:
AgentConfig(name='top_ppo',
            communication_space=None,
            agent_class=<class 'tame.agents.monolithic_ppo.Agent'>,
            agent_kwargs={'args': Args(subargs=None,
                                       seed=1,
                                       cuda=0,
                                       exp_name='monolithic_ppo',
                                       torch_deterministic=True,
                                       save_model=True,
                                       total_timesteps=500000,
                                       learning_rate=0.00025,
                                       gamma=0.99,
                                       anneal_lr=True,
                                       gae_lambda=0.95,
                                       batch_size=2048,
                                       num_minibatches=4,
                                       update_epochs=4,
                                       norm_adv=True,

### Level

Finally we define the top level.
Here we won't have to define an action space, as this level does not expects actions from above.

Moreover, the environment of this level, is the level below.

In [12]:
top_level_config = LevelConfig(
                name="top",
                agents=[top_agent_config],
                uplinks=None, # No level above, so no uplinks
                downlinks=top_bottom_links,
                action_frequency=1,
                trace_type="full",
                env=hierarchy.levels[-1], # The environment that the agents in this level interact with. In this case the bottom level we defined before!
            )

In [13]:
# Finally, we add the level to the hierarchy
hierarchy.add_level_config(top_level_config)

Created level top with agents: ['top_ppo']


## Whole hierarchy

We can see the structure of the hierarchy through the `print_tree()` function, and all the details through `print_hierarchy_details()`.

In [14]:
hierarchy.print_tree()

└── top_ppo
    ├── agent_0
    │   └── agent_0 (env)
    └── agent_1
        └── agent_1 (env)


In [15]:
hierarchy.print_hierarchy_details()


=== Hierarchy Details ===

Level 0: bottom
────────────────────────────────────────────────────────────

Level Spaces:
  Observation Space: Dict('top_ppo': Dict('agent_0': Box(inf, inf, (12,), float32), 'agent_1': Box(inf, inf, (12,), float32)))
  Action Space: Dict('top_ppo': Dict('agent_0': Discrete(5), 'agent_1': Discrete(5)))

Uplinks:
  top_ppo → ['agent_0', 'agent_1']

Downlinks:
  agent_0 → ['agent_0']
  agent_1 → ['agent_1']

Agents:
  agent_0:
    Observation space: Dict('agent_0': Box(-inf, inf, (13,), float32))
    Action space: Dict('agent_0': Discrete(5))
    Communication space: Dict('agent_0': Box(inf, inf, (12,), float32))
  agent_1:
    Observation space: Dict('agent_1': Box(-inf, inf, (13,), float32))
    Action space: Dict('agent_1': Discrete(5))
    Communication space: Dict('agent_1': Box(inf, inf, (12,), float32))


Level 1: top
────────────────────────────────────────────────────────────

Level Spaces:
  Observation Space: Dict()
  Action Space: Dict()

Downlink

# Training and Evaluation

Training the hierarchy is simple, and is enough to call the `step()` function for as many `training_steps` desired.
This function will take care of stepping all the levels in the hierarchy, environment included, and training the agents by calling their `update_step` function.

In [16]:
# We evaluate the untrained agent first
from tame.utils.utils import evaluate

initial_reward = evaluate(agent=hierarchy, env=env, eval_runs=1)
initial_reward = initial_reward.sum()

Eval episode: 0 - TS: 100 - Total Reward:
agent_0   -109.396377
agent_1   -109.396377
dtype: float64


In [17]:
from tqdm import tqdm

# If you want to train with a new instance of the environment, you can pass it to the hierarchy like this.
# Otherwise the training will be done on the env that you passed to the bottom_level
hierarchy.connect(env)

# Training loop
done = False
episode = 0
hierarchy.reset()
total_timesteps = 10000
for global_step in tqdm(range(total_timesteps), desc="Training step:"):
    if done:
        done = False
        hierarchy.reset()
        episode += 1

    _, reward, terminated, truncated, _ = hierarchy.step(action=None)

    if any(terminated.values()) or any(truncated.values()):
        done = True

# Final cleanup
hierarchy.reset()

Training step::  20%|█▉        | 1995/10000 [00:04<00:19, 421.21it/s]

agent_0: Updating models...
agent_1: Updating models...
top_ppo: Updating models...


Training step::  40%|████      | 4010/10000 [00:11<00:13, 453.91it/s]

agent_0: Updating models...
agent_1: Updating models...
top_ppo: Updating models...


Training step::  61%|██████    | 6051/10000 [00:16<00:08, 459.42it/s]

agent_0: Updating models...
agent_1: Updating models...
top_ppo: Updating models...


Training step::  81%|████████  | 8069/10000 [00:22<00:04, 434.99it/s]

agent_0: Updating models...
agent_1: Updating models...
top_ppo: Updating models...


Training step:: 100%|██████████| 10000/10000 [00:27<00:00, 365.25it/s]


(None, {'top_ppo': {'agent_0': {}, 'agent_1': {}}})

In [18]:
# Reevaluate the agent after training
final_reward = evaluate(agent=hierarchy, env=env, eval_runs=1)
final_reward = final_reward.sum()

Eval episode: 0 - TS: 100 - Total Reward:
agent_0   -85.299738
agent_1   -85.299738
dtype: float64


Hopefully the final reward is better than the initial reward, meaning that your agent has learned!

In [23]:
print("\n================ Reward Summary ===============")
print(f"{'Agent':8} {'Initial':>12} {'Final':>12} {'Improvement':>12}")
print("-" * 44)

for agent in initial_reward.index:
    init = initial_reward[agent]
    fin = final_reward[agent]
    pct = ((fin - init) / abs(init)) * 100
    print(f"{agent:8} {init:12.2f} {fin:12.2f} {pct:>11.1f}%")


================ Reward Summary ===============
Agent         Initial        Final  Improvement
--------------------------------------------
agent_0       -109.40       -85.30        22.0%
agent_1       -109.40       -85.30        22.0%
